# Drift Detection — 02: Bayesian Online Changepoint Detection (BOCPD)

Instead of hand-coded signal patterns, BOCPD models the **posterior probability that a changepoint
occurred at each time step**. It maintains a "run length" distribution — how long the current
regime has been active — and updates it with each new observation.

The signal taxonomy (crash / fade / no-recovery) is **not an input**. These patterns emerge
from the posterior shape — a crash produces a sudden spike in P(changepoint); a fade produces
a gradual rise over multiple steps.

**Doc reference:** `docs/evolution/drift_detection.md § 3.2`

### Implementation

Uses a **Dirichlet-Categorical conjugate prior** for {-1, 0, +1} alignment scores.
- Prior: Dirichlet(α₀) — uninformative (α₀ = [1, 1, 1])
- Likelihood: Categorical(θ) — probability of each score value in the current regime
- Hazard rate: constant H (probability of changepoint at each step)
- At each step: update run-length distribution using Bayes' theorem

### Data note

With ~10-15 steps per persona per dimension, posterior estimates will be wide.
BOCPD is most powerful at ~50+ personas with confirmed changepoints. This notebook
validates the implementation and shows the qualitative behaviour on current data.


In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=0.9)

# --- Load annotations ---
files = {
    "JL":  "../../logs/annotations/jl.parquet",
    "KM":  "../../logs/annotations/km.parquet",
    "Des": "../../logs/annotations/des.parquet",
}

VALUE_COLS = [
    "alignment_self_direction", "alignment_stimulation", "alignment_hedonism",
    "alignment_achievement", "alignment_power", "alignment_security",
    "alignment_conformity", "alignment_tradition", "alignment_benevolence",
    "alignment_universalism",
]
SHORT_NAMES = [c.replace("alignment_", "") for c in VALUE_COLS]
DIM_LABELS  = ["SD", "ST", "HE", "AC", "PO", "SE", "CO", "TR", "BE", "UN"]

dfs = [pl.read_parquet(p) for p in files.values()]
raw = pl.concat(dfs)

registry   = pl.read_parquet("../../logs/registry/personas.parquet").select(
                ["persona_id", "name", "core_values"])
id_to_name = dict(zip(registry["persona_id"].to_list(), registry["name"].to_list()))
id_to_core = dict(zip(registry["persona_id"].to_list(), registry["core_values"].to_list()))

mean_df = (
    raw.group_by(["persona_id", "t_index"])
    .agg([pl.col(c).cast(pl.Float64).mean().alias(c) for c in VALUE_COLS])
    .sort(["persona_id", "t_index"])
)
persona_ids = sorted(mean_df["persona_id"].unique().to_list())
print(f"Loaded {len(raw)} annotations across {len(persona_ids)} personas")


def get_persona_matrix(pid: str) -> tuple[list[int], np.ndarray]:
    pdata = mean_df.filter(pl.col("persona_id") == pid).sort("t_index")
    return pdata["t_index"].to_list(), np.array([pdata[c].to_list() for c in VALUE_COLS]).T


def get_profile_weights(pid: str) -> np.ndarray:
    core = id_to_core.get(pid, [])
    name_to_idx = {s.lower().replace("-", "_"): i for i, s in enumerate(SHORT_NAMES)}
    w = np.zeros(10)
    for v in core:
        key = v.lower().replace("-", "_").replace(" ", "_")
        if key in name_to_idx:
            w[name_to_idx[key]] = 1.0
    if w.sum() > 0:
        w /= w.sum()
    return w


def core_dim_indices(weights: np.ndarray, w_min: float = 0.15) -> list[int]:
    return [j for j, wj in enumerate(weights) if wj >= w_min]


## BOCPD Implementation

Dirichlet-Categorical conjugate update for ordinal {-1, 0, +1} scores.
The run-length posterior `R[t]` is a distribution over how many steps the current regime has lasted.

In [ ]:
def bocpd_dimension(
    scores_1d: np.ndarray,
    hazard: float = 0.1,
    alpha0: np.ndarray | None = None,
) -> dict:
    """
    Bayesian Online Changepoint Detection with Dirichlet-Categorical likelihood.

    Parameters
    ----------
    scores_1d : (T,) array of alignment scores in {-1, 0, +1}
    hazard    : prior probability of changepoint at each step
    alpha0    : Dirichlet prior pseudo-counts for bins [-1, 0, +1]. Default [1,1,1].

    Returns
    -------
    dict with keys:
        p_change : (T,) array — P(changepoint at t | data_{1:t})
        run_probs: list of (T,) arrays — run-length posterior at each step
    """
    if alpha0 is None:
        alpha0 = np.array([1.0, 1.0, 1.0])

    T = len(scores_1d)

    def score_to_bin(s: float) -> int:
        """Map {-1, 0, +1} (possibly fractional means) to bin index."""
        if s < -0.5: return 0
        if s <  0.5: return 1
        return 2

    # R[t] is a list of (run_length, alpha) tuples — one per hypothetical run
    # Start with a single run of length 0 at t=0
    runs = [(0.0, alpha0.copy())]   # (log_prob, alpha)
    # We track unnormalised log-probabilities for numerical stability
    log_weights = [0.0]             # log P(run_length = 0 at t=0)

    p_change = np.zeros(T)

    for t in range(T):
        obs_bin = score_to_bin(scores_1d[t])
        n_runs = len(runs)

        new_runs = []
        new_log_weights = []

        # Changepoint: new run starts (run_length = 0)
        cp_log_weight = -np.inf

        for i, ((rl, alpha), lw) in enumerate(zip(runs, log_weights)):
            total = alpha.sum()
            log_pred = np.log(alpha[obs_bin] / total)   # predictive under this run
            updated_alpha = alpha.copy()
            updated_alpha[obs_bin] += 1

            # Probability of continuing this run (no changepoint)
            log_continue = lw + log_pred + np.log(1 - hazard)
            new_runs.append((rl + 1, updated_alpha))
            new_log_weights.append(log_continue)

            # Probability of changepoint (new run) — accumulate
            log_cp_contrib = lw + log_pred + np.log(hazard)
            if cp_log_weight == -np.inf:
                cp_log_weight = log_cp_contrib
            else:
                m = max(cp_log_weight, log_cp_contrib)
                cp_log_weight = m + np.log(np.exp(cp_log_weight - m) + np.exp(log_cp_contrib - m))

        # Add the new run (changepoint at t)
        new_runs.append((0, alpha0.copy()))
        new_log_weights.append(cp_log_weight)

        # Normalise
        max_lw = max(new_log_weights)
        weights_unnorm = np.array([np.exp(lw - max_lw) for lw in new_log_weights])
        weights_norm = weights_unnorm / weights_unnorm.sum()

        # P(changepoint at t) = weight of the new run (last entry)
        p_change[t] = weights_norm[-1]

        runs = new_runs
        log_weights = [np.log(max(w, 1e-300)) for w in weights_norm]

    return {"p_change": p_change}


## Run BOCPD on All Personas

For each persona × core dimension, compute P(changepoint at t) across all time steps.
Alert when P(changepoint) > threshold.

In [ ]:
HAZARD    = 0.15    # prior probability of changepoint per step
CP_THRESH = 0.5     # alert when P(changepoint) exceeds this
W_MIN     = 0.15

bocpd_results = {}

for pid in persona_ids:
    t_idx, matrix = get_persona_matrix(pid)
    T = len(t_idx)
    if T < 5:
        continue

    w = get_profile_weights(pid)
    core_j = core_dim_indices(w, W_MIN)

    dim_results = {}
    alerts = []

    for j in core_j:
        out = bocpd_dimension(matrix[:, j], hazard=HAZARD)
        dim_results[j] = out
        for t, pc in enumerate(out["p_change"]):
            if pc > CP_THRESH:
                alerts.append((t, j))

    bocpd_results[pid] = {
        "name":       id_to_name.get(pid, pid[:8]),
        "core":       id_to_core.get(pid, []),
        "T":          T,
        "t_idx":      t_idx,
        "matrix":     matrix,
        "weights":    w,
        "dim_results": dim_results,
        "alerts":     alerts,
    }

print(f"BOCPD ran on {len(bocpd_results)} personas\n")
print(f"{'Persona':<25s}  {'Core':<28s}  T  Alerts")
print("-" * 70)
for pid, r in bocpd_results.items():
    print(f"{r['name']:<25s}  {', '.join(r['core']):<28s}  {r['T']:2d}  {len(r['alerts'])}")


## Visualise P(changepoint) Over Time

For each persona, show alignment scores on core dimensions alongside P(changepoint).
Red shading = P(changepoint) > threshold.

In [ ]:
palette = sns.color_palette("tab10", n_colors=10)


def plot_bocpd_persona(pid: str):
    r = bocpd_results[pid]
    matrix = r["matrix"]
    T = r["T"]
    core_j = [j for j in r["dim_results"]]
    if not core_j:
        return

    fig, axes = plt.subplots(len(core_j), 2, figsize=(14, 2.5 * len(core_j)),
                             sharex=True, gridspec_kw={"width_ratios": [2, 1]})
    if len(core_j) == 1:
        axes = [axes]

    for ax_row, j in zip(axes, core_j):
        ax_score, ax_cp = ax_row
        p_change = r["dim_results"][j]["p_change"]

        # Score plot
        ax_score.plot(range(T), matrix[:, j], "o-", color=palette[j], lw=1.8)
        ax_score.axhline(0, color="grey", lw=0.5, ls="--")
        ax_score.set_ylim(-1.4, 1.4)
        ax_score.set_ylabel(DIM_LABELS[j], fontsize=9)

        # Shade high P(changepoint) regions
        for t, pc in enumerate(p_change):
            if pc > CP_THRESH:
                ax_score.axvspan(t - 0.4, t + 0.4, color="red", alpha=pc * 0.4, zorder=1)

        # P(changepoint) plot
        ax_cp.bar(range(T), p_change, color="salmon", alpha=0.8)
        ax_cp.axhline(CP_THRESH, color="red", lw=1, ls="--")
        ax_cp.set_ylim(0, 1)
        ax_cp.set_ylabel("P(change)", fontsize=8)
        ax_cp.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))

    axes[0][0].set_title(f"{r['name']} — {', '.join(r['core'])}", fontweight="bold")
    axes[-1][0].set_xlabel("t_index")
    axes[-1][1].set_xlabel("t_index")
    plt.tight_layout()
    plt.show()


for pid in list(bocpd_results.keys())[:4]:
    plot_bocpd_persona(pid)


## Sensitivity Analysis: Hazard Rate

The hazard rate H controls how often the model expects changepoints a priori.
Low H = conservative (needs strong evidence). High H = sensitive (fires more often).

In [ ]:
hazard_values = [0.05, 0.10, 0.15, 0.25, 0.40]

# Pick first persona with ≥6 steps and ≥1 core dimension
example_pid = next(
    pid for pid, r in bocpd_results.items()
    if r["T"] >= 6 and r["dim_results"]
)
example_r = bocpd_results[example_pid]
example_j = list(example_r["dim_results"].keys())[0]
scores_1d  = example_r["matrix"][:, example_j]
T          = example_r["T"]

fig, axes = plt.subplots(len(hazard_values), 1,
                         figsize=(12, 2.2 * len(hazard_values)), sharex=True)

for ax, H in zip(axes, hazard_values):
    out = bocpd_dimension(scores_1d, hazard=H)
    ax.bar(range(T), out["p_change"], color="salmon", alpha=0.8, label=f"H={H}")
    ax.axhline(CP_THRESH, color="red", lw=1, ls="--")
    ax.set_ylim(0, 1)
    ax.set_ylabel(f"H={H}", fontsize=9)
    ax2 = ax.twinx()
    ax2.plot(range(T), scores_1d, "o-", color="steelblue", ms=4, lw=1.2)
    ax2.set_ylim(-1.4, 1.4)
    ax2.set_ylabel("score", fontsize=8)

axes[0].set_title(
    f"{example_r['name']} — {DIM_LABELS[example_j]}: P(changepoint) vs hazard rate",
    fontweight="bold")
axes[-1].set_xlabel("t_index")
plt.tight_layout()
plt.show()


## Limitations

- **Small data:** ~10-15 steps per persona per dimension. Posterior is prior-dominated early on.
- **Discrete scores:** {-1, 0, +1} means regime estimates are noisy on short windows.
- **No profile weights `w_u` natively:** BOCPD operates per-dimension independently. Profile weighting requires a wrapper (e.g., suppress alerts on low-weight dimensions).
- **Interpretability:** `P(changepoint) = 0.87` needs translation to user-facing language for the Coach.

At ~50+ personas with confirmed changepoints, BOCPD becomes the recommended upgrade from rule-based (see `docs/evolution/drift_detection.md § 3.2`).